# LatentDriver Colab Runner

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/achyutmorang/latentdriver-waymax-experiments/blob/main/notebooks/latentdriver_colab_runner.ipynb)

Use this notebook as a terminal-style launcher. Notebook cells are shell commands; clone/pull, Drive binding, and execution logic live in `scripts/*.py` and `src/`.


## Operating Model

1. Run the Drive mount code cell. This is a Colab platform exception, not experiment logic.
2. Run the GCS auth code cell if you want to evaluate directly from `gs://waymo_open_dataset_motion_v_1_1_0`. Skip it only if you intentionally use the Drive-local `assets/raw_womd` cache.
3. Run the bootstrap shell cell.
4. Optionally list profiles.
5. Use the profile shell cell as your terminal launcher.
6. Inspect the debug status shell cell if a profile fails.

Only the first two code cells are notebook-native Python. Every other cell stays shell-first so the runnable logic remains in `scripts/` and `src/`.


In [ ]:
from pathlib import Path

DRIVE_MOUNTPOINT = "/content/drive"
drive_root = Path(DRIVE_MOUNTPOINT) / "MyDrive"

if drive_root.is_dir():
    print(f"Drive already mounted at {drive_root}")
else:
    from google.colab import drive
    drive.mount(DRIVE_MOUNTPOINT)
    print(f"Mounted Drive at {drive_root}")


In [ ]:
from __future__ import annotations

import json
import subprocess

USE_GCS_DIRECT = True

if USE_GCS_DIRECT:
    from google.colab import auth
    auth.authenticate_user()
    checks = {}
    commands = {
        "active_account": ["gcloud", "auth", "list", "--filter=status:ACTIVE", "--format=json"],
        "adc_token": ["gcloud", "auth", "application-default", "print-access-token"],
    }
    for name, command in commands.items():
        proc = subprocess.run(command, text=True, capture_output=True, check=False)
        payload = {"command": command, "returncode": proc.returncode}
        if name == "adc_token":
            payload["token_ready"] = proc.returncode == 0 and bool(proc.stdout.strip())
            payload["stderr"] = proc.stderr.strip()
        else:
            payload["stdout"] = proc.stdout.strip()
            payload["stderr"] = proc.stderr.strip()
        checks[name] = payload
    print(json.dumps({"use_gcs_direct": True, "checks": checks}, indent=2))
else:
    print(json.dumps({
        "use_gcs_direct": False,
        "message": "Skipping GCS auth because this session will use the Drive-local assets/raw_womd cache.",
    }, indent=2))


In [ ]:
%%bash
set -euo pipefail

BOOTSTRAP=/tmp/latentdriver_colab_bootstrap.py
curl -fsSL \
  https://raw.githubusercontent.com/achyutmorang/latentdriver-waymax-experiments/main/scripts/colab_bootstrap.py \
  -o "$BOOTSTRAP"

python3 "$BOOTSTRAP" \
  --repo-url https://github.com/achyutmorang/latentdriver-waymax-experiments.git \
  --branch main \
  --repo-dir /content/latentdriver-waymax-experiments \
  --drive-base-root /content/drive/MyDrive/waymax_research \
  --waymo-dataset-root gs://waymo_open_dataset_motion_v_1_1_0


In [ ]:
%%bash
set -euo pipefail
cd /content/latentdriver-waymax-experiments
python3 scripts/colab_canary.py --list-profiles


In [ ]:
%%bash
set -euo pipefail
cd /content/latentdriver-waymax-experiments

# Default path: authenticate GCS above, then dry-run against the gs:// WOMD validation root.
python3 scripts/colab_canary.py \
  --profile full-eval-dry-run \
  --model latentdriver_t2_j3 \
  --seed 0 \
  --vis video \
  --auto-install-runtime \
  --waymo-dataset-root gs://waymo_open_dataset_motion_v_1_1_0

# Fallback path if direct GCS auth is unavailable or TensorFlow still cannot read gs:// in this runtime.
# python3 scripts/colab_canary.py \
#   --profile stage-full-womd-validation \
#   --waymo-dataset-root gs://waymo_open_dataset_motion_v_1_1_0
#
# python3 scripts/colab_canary.py \
#   --profile full-eval-dry-run \
#   --model latentdriver_t2_j3 \
#   --seed 0 \
#   --vis video \
#   --auto-install-runtime \
#   --waymo-dataset-root /content/latentdriver-waymax-experiments/artifacts/assets/raw_womd

# After the dry-run reports ready: true, switch to the real eval profile.
# python3 scripts/colab_canary.py \
#   --profile full-eval-reactive-single \
#   --model latentdriver_t2_j3 \
#   --seed 0 \
#   --vis video \
#   --auto-install-runtime \
#   --waymo-dataset-root gs://waymo_open_dataset_motion_v_1_1_0


In [ ]:
%%bash
set -euo pipefail
cd /content/latentdriver-waymax-experiments

printf '\n--- debug root ---\n'
ls -lah results/debug_runs || true

printf '\n--- current latest run: LATEST.json ---\n'
cat results/debug_runs/LATEST.json || true

printf '\n--- historical latest failure: LATEST_FAILURE.json ---\n'
printf 'This is only the most recent failed run, not necessarily the current run. Trust LATEST.json for current status.\n'
if [ -f results/debug_runs/LATEST_FAILURE.json ]; then
  cat results/debug_runs/LATEST_FAILURE.json
else
  printf 'No failure pointer recorded yet.\n'
fi

printf '\n--- stable aliases ---\n'
find results/debug_runs -maxdepth 2 -type f \
  \( -name 'ALIAS.json' -o -name 'manifest.json' -o -name 'failure_summary.json' \) \
  -print | sort || true
